# Drought Category Reality Check: the evidence

**Team Wolverine** · Data to Harvest Hackathon, DSU 2026

**User:** corn and soybean farmers in Iowa and Nebraska.
**Decision:** how much yield loss to plan for when their county reaches a drought level such as D3, and whether a D3 costs the same in both states.

This notebook reproduces every number in the README and the app, using the team's own code in `wolverine/`:
1. Build a drought level → yield loss table on **2000–2012**.
2. Check it on **2013–2025**, years it never saw.
3. Explain the misses: the 2012 drought, drought timing and irrigation.

In [ ]:
import sys
sys.path.insert(0, "..")   # run from notebooks/, import from the repo root

import numpy as np
import pandas as pd
import plotly.express as px

from wolverine import data_prep, reality_table as rt, drought_timing as dt
from wolverine.contracts import *

pd.set_option("display.precision", 1)

## 0. The panel

Master dataset rows for Iowa and Nebraska, corn and soybeans, with a NASS-reported yield and a drought reading. `anom_strict` is a second yield anomaly whose trend is fitted on 2000–2012 only.

In [ ]:
panel = data_prep.add_strict_anomaly(data_prep.build_panel())
print(f"{len(panel):,} county-crop-years, {panel.fips.nunique()} counties, "
      f"{panel.year.min()}-{panel.year.max()}")
print(f"anom_strict missing for {panel[STRICT_ANOM].isna().sum()} rows (county-crops with < 10 training years)")
panel.head()

In [ ]:
# Rows per state, crop and drought level, in the training and test periods.
# max_drought_severity: 0 = none, 1 = D0 ... 4 = D3, 5 = D4
data_prep.coverage_report(panel)

## 1. The table: yield loss by drought level, 2000–2012

Each cell is the mean yield anomaly (% below or above the county's trend) for one state, crop and drought level. Negative = loss. Main result uses the master trend (`yield_anomaly_pct`).

In [ ]:
table = rt.build_table(panel, TRAIN_YEARS)
table.pivot_table(index="level_label", columns=["crop", "state_alpha"], values="mean_loss", sort=False)

In [ ]:
fig = px.bar(table, x="level_label", y="mean_loss", color="state_alpha", facet_col="crop",
             barmode="group", hover_data=["n"],
             category_orders={"level_label": list(LEVEL_LABELS.values())},
             labels={"level_label": "Drought level", "mean_loss": "Mean yield anomaly (%)", "state_alpha": "State"},
             title="Yield loss by drought level, 2000–2012 (master trend)")
fig.show()

In [ ]:
# Does D3 cost the same in both states? (within 4 points = same)
pd.DataFrame({crop: rt.same_meaning(table, level=4, crop=crop) for crop in CROPS}).T

**Finding:** D3 cost Iowa corn 17.4% and Nebraska corn 6.0%, an 11.4-point gap. Soybeans: 15.1% vs 9.9%. The same label does not mean the same loss.

## 2. Did it hold up on 2013–2025?

A cell holds up if its 2013–2025 mean is within 4 points of its 2000–2012 mean. Cells need at least 10 rows in both periods to be scored. We score each crop on both trends: master (main result) and strict (robustness check).

In [ ]:
rows = []
for crop in CROPS:
    for name, col in [("master (main result)", "yield_anomaly_pct"), ("strict (robustness check)", STRICT_ANOM)]:
        check = rt.backtest(panel[panel.crop == crop], col)
        s = rt.score(check)
        rows.append({"crop": crop, "trend": name,
                     "cells within 4 pts": f"{s['share_within_tol']:.0%}",
                     "weighted mean error (pts)": round(s["weighted_mae"], 1),
                     "cells scored": f"{s['cells_scored']} of {len(check)}"})
pd.DataFrame(rows)

In [ ]:
# Cell by cell, corn, master trend (the app's default view)
check = rt.backtest(panel[panel.crop == "CORN"], "yield_anomaly_pct")
check[["state_alpha", "level_label", "train_mean", "train_n", "test_mean", "test_n",
       "abs_error", "within_tol", "enough_rows"]].round(1)

In [ ]:
ok = check.dropna(subset=["train_mean", "test_mean"])
lo, hi = ok[["train_mean", "test_mean"]].min().min() - 2, ok[["train_mean", "test_mean"]].max().max() + 2
fig = px.scatter(ok, x="train_mean", y="test_mean", color="state_alpha", text="level_label",
                 labels={"train_mean": "2000–2012 mean (%)", "test_mean": "2013–2025 mean (%)"},
                 title="Corn: training vs test mean per cell (dashed = perfect, grey = ±4 pts)")
fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(dash="dash", color="grey"))
fig.add_shape(type="path", path=f"M {lo},{lo+4} L {hi},{hi+4} L {hi},{hi-4} L {lo},{lo-4} Z",
              fillcolor="rgba(128,128,128,0.15)", line_width=0)
fig.update_traces(textposition="top center")
fig.show()

**Finding:** on the master trend 82% of scored cells held up, for corn and for soybeans. The biggest miss is **Iowa at D3**: −17.4% in training, −3.9% in testing.

## 3. Why Iowa D3 misses: the 2012 drought

### 3a. 2012 dominates the training rows

In [ ]:
train = panel[panel.year.between(*TRAIN_YEARS) & (panel.max_drought_severity >= 4)]
share_2012 = (train.assign(is_2012=train.year == 2012)
                   .groupby(["state_alpha", "crop"]).is_2012.agg(["mean", "size"])
                   .rename(columns={"mean": "share from 2012", "size": "D3+ training rows"}))
share_2012.style.format({"share from 2012": "{:.0%}"})

### 3b. Why the strict trend breaks for Iowa corn

The strict trend is fitted on 2000–2012, which ends on the 2012 drought. That pulls each county's slope down, so the line extended into 2013–2025 sits too low and later years look better than they were.

In [ ]:
def slope(g):
    return np.polyfit(g.year, g.yield_per_acre, 1)[0]

ia_corn = panel[(panel.state_alpha == "IA") & (panel.crop == "CORN") & panel.year.between(*TRAIN_YEARS)]
with_2012 = ia_corn.groupby("fips").apply(slope)
without_2012 = ia_corn[ia_corn.year != 2012].groupby("fips").apply(slope)
print(f"Median Iowa corn trend slope, 2000-2012: {with_2012.median():.2f} bu/ac/yr with 2012, "
      f"{without_2012.median():.2f} without it")

test = panel[panel.year.between(*TEST_YEARS)]
test.groupby(["state_alpha", "crop"])[STRICT_ANOM].mean().rename("mean strict anomaly, 2013-2025 (%)")

Iowa corn's strict anomalies average about +23% in the test years, while the other three state-crops sit near 0. That is why corn scores 55% on the strict trend.

## 4. Why the states differ: timing and irrigation

### 4a. Drought timing (D3 seasons, 2000–2025)

`peak` = severe drought (D2 or worse) in July–August; `early` = only before July; `late` = only after August.

In [ ]:
timed = panel.merge(dt.drought_timing(), on=["fips", "crop", "year"], how="left")
t = rt.build_table(timed, (TRAIN_YEARS[0], TEST_YEARS[1]), extra_by=["timing"])
t[t.level == 4].pivot_table(index=["crop", "state_alpha"], columns="timing", values=["mean_loss", "n"])

Almost every D3 season includes peak-season drought. Early-only D3 is rare (9 county-years for Iowa corn), so those cells are too thin to quote.

### 4b. Irrigation (Nebraska, 2000–2012)

Each Nebraska county gets one average irrigated share (data runs 2000–2018) and is split at 50%.

In [ ]:
groups = dt.irrigation_groups()
print(groups.groupby(["crop", "irrig_group"]).size().rename("counties").to_string())

ne = panel[panel.state_alpha == "NE"].merge(groups, on=["fips", "crop"], how="left")
irr = rt.build_table(ne, TRAIN_YEARS, extra_by=["irrig_group"])
irr.pivot_table(index="level_label", columns=["crop", "irrig_group"], values="mean_loss", sort=False)

In [ ]:
fig = px.bar(irr, x="level_label", y="mean_loss", color="irrig_group", facet_col="crop",
             barmode="group", hover_data=["n"],
             category_orders={"level_label": list(LEVEL_LABELS.values())},
             labels={"level_label": "Drought level", "mean_loss": "Mean yield anomaly (%)", "irrig_group": "County group"},
             title="Nebraska: yield loss by drought level, irrigated vs dryland, 2000–2012")
fig.show()

**Finding:** at D3, dryland Nebraska corn lost 17.8% and irrigated corn 0.9% (soybeans 17.6% vs 4.6%). Dryland Nebraska loses about what Iowa does, so irrigation explains most of the gap between the states.

## 5. The recommendation

In [ ]:
for crop in CROPS:
    v = panel[panel.crop == crop]
    print(f"{crop}: " + rt.recommendation(rt.backtest(v), rt.build_table(v, TRAIN_YEARS)))

## Conclusion

- **D3 does not mean the same in Iowa and Nebraska.** Don't read the drought map the same way in every state: at D3, plan for about a 17% yield loss on Iowa corn but about 6% in Nebraska (much less on irrigated land).
- **The table mostly holds up:** 82% of scored cells within 4 points on 2013–2025 (master trend), for both crops.
- **Where it misses, we know why:** Iowa's training D3 is mostly the 2012 drought; Nebraska's smaller losses come from irrigation.
- **Caveats:** the master trend sees the test years; the strict trend is distorted for Iowa corn by 2012; Iowa D4 cells are too thin to score. See the README's Limitations section.